In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv  
import os   

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

/Users/admin/repos/geneforge


In [2]:
import random
from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow
OUTPUT_DIR = "datasets/maximize_promoter_strength_dataset"

def get_promoters():
    from src.library.cello_library import CelloLibrary
    library = CelloLibrary()
    library.select_library("Eco1C1G1T1")
    promoters = {collection["name"]: collection["dnasequence"] for collection in library.user_constraints if collection["collection"] == "parts" and collection["type"] == "promoter"}
    return promoters

promoter_pool = get_promoters()
promoter_names = list(promoter_pool.keys())
random.shuffle(promoter_names)

/Users/admin/repos/geneforge/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:library_manager:Library manager initialized. Found 5 potential libraries.


In [ ]:
promoter_sequence = promoter_pool[promoter_names[0]]
print(promoter_sequence)
workflow = MaximizePromoterStrengthWorkflow(example_name="maximize_promoter_strength_workflow_test",
                                           promoter_sequence=promoter_sequence,
                                           use_reasoning_model=True)
workflow.run() 
print(workflow.get_metrics())

In [ ]:
workflow.messages

In [3]:
from sklearn.model_selection import train_test_split
from src.examples.agent.egc_problem1p1 import EGCProblem1p1Workflow
from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

message_lists = []
metrics = []
scenarios = []

train_sequences, eval_sequences = train_test_split(list(promoter_pool.keys()), test_size=0.2, random_state=42)

print('Eval sequences: ', len(eval_sequences))
print('Train sequences: ', len(train_sequences))

# for promoter_name in eval_sequences:
#     promoter_sequence = promoter_pool[promoter_name]
    
#     run_name = "maximize_promoter_strength" 
#     for run_index in range(1):
#         run_id = f"{run_name}_{promoter_name}_{run_index}"
        
#         workflow = MaximizePromoterStrengthWorkflow(example_name=run_id,
#                                                     promoter_sequence=promoter_sequence,
#                                                     use_reasoning_model=True)
        
#         scenarios.append(workflow)
        
# scenarios += [EGCProblem1p1Workflow(example_name="egc_problem1p1_workflow_test")]

Eval sequences:  3
Train sequences:  9


In [19]:
from src.adapters.art_adapter import ArtAdapter
import art
from art.gather import gather_trajectory_groups
from src.examples.agent.egc_problem1p1 import EGCProblem1p1Workflow
from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

training_config = {
    "groups_per_step": 2,
    "num_epochs": 20,
    "rollouts_per_group": 3,
    "learning_rate": 1e-5,
    "max_steps": 20,
}

step = 0
max_promoter_strength_groups = await gather_trajectory_groups(
    (
            art.TrajectoryGroup((
                ArtAdapter(MaximizePromoterStrengthWorkflow(example_name=f"maximize_promoter_strength_workflow_test_{promoter_sequence}",
                                                             promoter_sequence=promoter_sequence,
                                                             use_reasoning_model=True), 
                            step=step).rollout()
                    for promoter_sequence in eval_sequences
                for _ in range(training_config["rollouts_per_group"]))
            ),
    ),
    pbar_desc="gather",
    max_exceptions=18,)

INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
gather:  11%|█         | 1/9 [00:04<00:35,  4.47s/it, reward=0]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

INFO:src.integrations.pro_d_integration:Evaluating 1 spacer sequences with ProD
INFO:src.integrations.pro_d_integration:Evaluating 1 spacer sequences with ProD
gather:  22%|██▏       | 2/9 [01:22<05:34, 47.71s/it, reward=0, answer_strength=2.84, reference_strength=2.84, difference=0, reference_class=5, answer_class=5, sequence_similarity=1, num_rounds=19]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

gather:  33%|███▎      | 3/9 [01:39<03:23, 33.90s/it, reward=0, answer_strength=2.84, reference_strength=2.84, difference=0, reference_class=5, answer_class=5, sequence_similarity=1, num_rounds=19]

...

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

INFO:src.integrations.pro_d_integration:Evaluating 1 spacer sequences with ProD
INFO:src.integrations.pro_d_integration:Evaluating 1 spacer sequences with ProD
gather:  89%|████████▉ | 8/9 [02:47<00:16, 16.68s/it, reward=0, answer_strength=2.76, reference_strength=2.76, difference=0, reference_class=6, answer_class=6, sequence_similarity=0.667, num_rounds=22.7]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

gather: 100%|██████████| 9/9 [03:30<00:00, 23.43s/it, reward=0, answer_strength=2.76, reference_strength=2.76, difference=0, reference_class=6, answer_class=6, sequence_similarity=0.667, num_rounds=22.7]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

In [25]:
# RUN THIS. SHOULD NOT SEE THE ValidatorIterator or whatever it was
max_promoter_strength_groups[0].trajectories[2].messages_and_choices

[{'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of the library by querying for pa

In [ ]:
promoter_sequence = train_sequences[0]
workflow = MaximizePromoterStrengthWorkflow(example_name=f"maximize_promoter_strength_workflow_test_{promoter_sequence}",
                                                             promoter_sequence=promoter_sequence,
                                                                 use_reasoning_model=True)
workflow.run()

In [28]:
from src.adapters.art_adapter import ArtAdapter
import art
from art.gather import gather_trajectory_groups
from src.examples.agent.egc_problem1p1 import EGCProblem1p1Workflow
from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow
training_config = {
    "groups_per_step": 2,
    "num_epochs": 20,
    "rollouts_per_group": 5,
    "learning_rate": 1e-5,
    "max_steps": 20,
    "max_rounds": 30,
}

step = 0
egc_problem_1_groups = await gather_trajectory_groups(
    (
            art.TrajectoryGroup(
                (ArtAdapter(EGCProblem1p1Workflow(example_name=f"egc_problem_1_workflow_test",
                                                  use_reasoning_model=False), 
                            step=step).rollout(max_rounds=training_config["max_rounds"]) 
                 for _ in range(training_config["rollouts_per_group"]))
            ),
            
    ),
    pbar_desc="gather",
    max_exceptions=18,)

INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
INFO:library_manager:Library manager initialized. Found 5 potential libraries.
gather:  20%|██        | 1/5 [00:06<00:24,  6.16s/it, reward=0]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

gather:  40%|████      | 2/5 [00:07<00:10,  3.34s/it, reward=0]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

gather:  60%|██████    | 3/5 [00:08<00:04,  2.14s/it, reward=0]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

gather:  80%|████████  | 4/5 [00:15<00:04,  4.01s/it, reward=0]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

gather: 100%|██████████| 5/5 [00:32<00:00,  6.53s/it, reward=0]

Messages: [{'role': 'system', 'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of th

In [29]:
for i, trajectory in enumerate(egc_problem_1_groups[0].trajectories):
    print(i)
    for m in trajectory.messages_and_choices:
        print(m)
    trajectory.model_copy(deep=True)


0
{'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of the library by querying for p

In [ ]:
from art.rewards import ruler
from art import TrainConfig
from art.rewards import ruler_score_group

from src.examples.agent.egc_problem1p1 import RUBRIC as RUBRIC_EGC_PROBLEM_1
from src.examples.agent.maximize_promoter_strength import GRADING_RUBRIC as RUBRIC_MAXIMIZE_PROMOTER_STRENGTH

judged_groups = []


for egc_problem_1_group in egc_problem_1_groups:
    # Use RULER to assign relative scores to each trajectory
    rubric = RUBRIC_EGC_PROBLEM_1
    judged_group = await ruler_score_group(egc_problem_1_group, "openai/o4-mini", debug=True)
    judged_groups.append(judged_group)
    
for group in max_promoter_strength_groups:
    
    # 1. We can score ourselves 
    for trajectory in group.trajectories:
        reward = MaximizePromoterStrengthWorkflow.score_trajectory(trajectory)
        trajectory.metadata["reward"] = reward
        judged_groups.append(trajectory)
    
    # 2. We can also use RULER to score the group
    rubric = RUBRIC_MAXIMIZE_PROMOTER_STRENGTH
    judged_group = await ruler_score_group(group, "openai/o4-mini", debug=True)
    judged_groups.append(judged_group)
        

# Now that we have computed the reward, we can train the model on all the judged groups
# await model.delete_checkpoints()
# await model.train(
#     judged_groups,
#     config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
#     # Lowering the logprob_calculation_chunk_size is a memory saving measure
#     # to allow longer sequences (up to 8192 tokens) to be processed on a T4.
#     _config={"logprob_calculation_chunk_size": 8},
# )

11:39:56 - LiteLLM:INFO: utils.py:3227 - 
LiteLLM completion() model= o4-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= o4-mini; provider = openai
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[RULER] Pretty-printed LLM choice JSON:

{
    'scores': [
        {
            'trajectory_id': '1',
            'explanation': 'The answer is entirely incorrect: the ΔG sign and magnitude are wrong and the 
steady‐state concentrations do not satisfy ΔG=0.',
            'score': 0.2
        },
        {
            'trajectory_id': '2',
            'explanation': 'ΔG is incorrectly set to zero, reaction direction is wrong, and the steady‐state values
are incorrect.',
            'score': 0.1
        },
        {
            'trajectory_id': '3',
            'explanation': 'Both the calculated ΔG and reaction direction are wrong and the proposed steady‐state 
concentrations do not achieve ΔG=0.',
            'score': 0.15
        },
        {
            'trajectory_id': '4',
            'explanation': 'Correct approach and ΔG sign/magnitude and reaction direction are right, but the 
steady‐state concentrations are numerically incorrect.',
            'score': 0.5
        },
        {
            'trajectory_id': '5',
            'explanation': 'ΔG sign, reaction direction, and steady‐state concentrations are all incorrect.',
            'score': 0.05
        }
    ]
}

In [ ]:
from art.rewards import ruler
from art.rewards import ruler_score_group
max_promoter_strength_groups_judged = await ruler_score_group(max_promoter_strength_groups[0], "openai/o4-mini", debug=True)

max_promoter_strength_groups_judged